# 07 · A transformer challenger: NILMFormer vs Seq2Point

**Requirement 3, extended.** Seq2Point is a 2018 convolutional design. Before standing behind it,
I tested it against the strongest recent transformer I could reproduce faithfully, under the
same data, homes, masks and metrics.

| | |
|---|---|
| **Input** | the same cleaned 1-minute corpus and split as notebooks 03–06; Seq2Point checkpoints `m2_seq2point_w81_s*` |
| **Output** | checkpoints `artifacts/models/m5_nilmformer_l129_s*`, `artifacts/tables/nb07_*.csv`, `artifacts/metrics/nb07_summary.json`, figures `nb07_*` |
| **Run time** | about one hour on one laptop GPU |

**Why NILMFormer.** NILMFormer (Petralia et al., KDD 2025) was designed for the problem that hurts
this project most: household load is non-stationary, so a model trained on some homes sees
different load levels in others. It standardises every window and feeds the window's mean and
spread back as a separate token, so the output scale is relearned per window. It also gets the
time of day, day of week and month as inputs. It is the only transformer I found with public code
(Apache-2.0) and published 1-minute REFIT washing-machine results. Those results use different
homes and splits, so they are not comparable with mine. It is also small: about 0.38 M
parameters against 4.2 M for Seq2Point at an 81-minute window.

**Recipe.** I follow the published recipe and declare every deviation:

| | published | here |
|---|---|---|
| architecture | 3 encoder layers, d = 96, 8 heads, dilated-conv embedding, stats token | same (port in `src/refit_nilm/models/nilmformer.py`, equivalence of the attention pinned by a test) |
| window | 128 / 256 / 512 minutes (128 best in the paper) | 129 (odd, so a window has a centre minute) |
| inputs | load + sin/cos of minute, hour, day of week, month | same, in UK local time |
| scaling | load and target divided by the largest training load | same |
| training | Adam 1e-4, batch 64, ≤ 50 epochs, early stop 10, LR on plateau 5, MSE | same; masked MSE so unusable minutes (`Issues`, outages) carry no loss, as for Seq2Point |
| inference | non-overlapping windows | each minute predicted by the window centred on it, exactly as Seq2Point is scored |

**Decision rule, fixed before any training** (committed to the repository before this notebook
was first run):

1. Single models, three seeds each (42, 10, 20), are compared on the validation home (House 18),
   on the minutes that both models can score.
2. The metrics are NDE and cycle F1, the two that selected the current model.
3. NILMFormer replaces Seq2Point only if its mean is better on **both** metrics by more than the
   larger of the two models' seed standard deviations. Otherwise Seq2Point stays: it is already
   validated, exported and served, and the challenger must clearly earn a change.
4. The test homes are then scored once for both three-seed ensembles, whatever the decision. This
   is the second time those homes are scored (the first was notebook 06), so I report the numbers
   and change nothing after seeing them.


In [ ]:
MAX_EPOCHS = 50  # the published budget; a quick end-to-end check of this notebook sets 1

In [ ]:
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from refit_nilm import config as C
from refit_nilm import datasets as D
from refit_nilm import metrics, plots
from refit_nilm import train as T
from refit_nilm import train_seq2seq as S

warnings.filterwarnings("ignore", category=FutureWarning)
plots.setup()
pd.set_option("display.width", 160, "display.max_columns", 30)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
W_S2P, W_NF = 81, 129
SEEDS = [42, 10, 20]
frames, corpora, stats = D.prepare([W_S2P, W_NF])
cp81, cp129, st81 = corpora[W_S2P], corpora[W_NF], stats[W_S2P]
SUMMARY: dict = {"window_seq2point": W_S2P, "window_nilmformer": W_NF, "seeds": SEEDS}
print({w: {k: len(v) for k, v in c.starts.items()} for w, c in corpora.items()})

## 1. The two models side by side

Parameter counts and the scaling constant. The training split is identical for both: the same
homes and the same first 80 % of each, so only the window length and the model differ.

In [ ]:
s2p = T.build("seq2point", W_S2P)
nf = S.build(S.Seq2SeqConfig())
n_s2p, n_nf = (sum(p.numel() for p in m.parameters()) for m in (s2p, nf))
scale = S.training_scale(cp129)
SUMMARY["parameters"] = {"seq2point": n_s2p, "nilmformer": n_nf}
SUMMARY["nilmformer_scale_w"] = scale
pd.DataFrame({"parameters": [n_s2p, n_nf], "window (min)": [W_S2P, W_NF], "output": ["centre minute", "every minute of the window"]},
             index=["Seq2Point", "NILMFormer"])

## 2. Training NILMFormer, three seeds

Each epoch draws as many random training windows as would tile the training minutes once (about
29,000 windows, 455 steps of 64). Early stopping watches validation NDE, as for Seq2Point.

In [ ]:
histories = {}
for seed in SEEDS:
    cfg = S.Seq2SeqConfig(window=W_NF, seed=seed, max_epochs=MAX_EPOCHS)
    t0 = time.time()
    model, hist, sc = S.fit(cp129, cfg, device=DEVICE)
    S.save_run(C.MODEL_DIR / f"m5_nilmformer_l{W_NF}_s{seed}.pt", model, cfg, sc, hist)
    histories[seed] = hist
    best = min(hist, key=lambda h: h["val_nde"])
    print(f"seed {seed}: {len(hist)} epochs, best epoch {best['epoch']} (val NDE {best['val_nde']:.3f}), {time.time() - t0:.0f}s")
    del model
    torch.cuda.empty_cache()
SUMMARY["nilmformer_epochs"] = {str(s): {"epochs": len(h), "best_epoch": min(h, key=lambda r: r["val_nde"])["epoch"]} for s, h in histories.items()}

In [ ]:
s2p_hist = {s: json.load(open(C.MODEL_DIR / f"m2_seq2point_w{W_S2P}_s{s}.json"))["history"] for s in SEEDS}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, (name, hs) in zip(axes, [("Seq2Point (W = 81)", s2p_hist), ("NILMFormer (L = 129)", histories)]):
    for s, h in hs.items():
        ax.plot([r["epoch"] for r in h], [r["val_nde"] for r in h], marker="o", ms=3, label=f"seed {s}")
    ax.set(title=name, xlabel="epoch", ylabel="validation NDE (House 18)")
    ax.legend()
fig.suptitle("Validation NDE by epoch (lower is better; the best epoch is kept)")
plots.save(fig, "nb07_learning_curves")
plt.show()

## 3. Validation comparison (the decision)

Both models are scored with the project's metric code on House 18, restricted to the minutes both
can score. A minute can be valid for an 81-minute window but not for a 129-minute one near an
outage, and vice versa, so the shared set is slightly smaller than either. Seq2Point's numbers on
its own full set are in notebook 05.

In [ ]:
S2P = {s: T.load_run(C.MODEL_DIR / f"m2_seq2point_w{W_S2P}_s{s}.pt", device=DEVICE)[0] for s in SEEDS}
NF = {s: S.load_run(C.MODEL_DIR / f"m5_nilmformer_l{W_NF}_s{s}.pt", device=DEVICE)[0] for s in SEEDS}
src81 = T.WindowSource(cp81, st81, DEVICE)
src129 = S.SeqWindowSource(cp129, scale, DEVICE)


def seq2point_power(cp, starts, seed):
    return T.predict(S2P[seed], src81, starts, st81)["power"]


def nilmformer_power(cp, starts, seed):
    return S.predict(NF[seed], src129, starts)


def series_for(split, house):
    # truth and per-seed predictions as minute series for one house, on the shared minutes
    st81, st129 = cp81.starts[split], cp129.starts[split]
    keep81 = cp81.house[st81 + W_S2P // 2] == house
    keep129 = cp129.house[st129 + W_NF // 2] == house
    st81, st129 = st81[keep81], st129[keep129]
    y = D.to_series(cp81, st81, cp81.wm[st81 + W_S2P // 2], house)
    preds = {}
    for s in SEEDS:
        preds[("Seq2Point", s)] = D.to_series(cp81, st81, seq2point_power(cp81, st81, s), house)
        preds[("NILMFormer", s)] = D.to_series(cp129, st129, nilmformer_power(cp129, st129, s), house)
    idx = y.index.union(preds[("NILMFormer", SEEDS[0])].index)
    y = y.reindex(idx)
    preds = {k: v.reindex(idx) for k, v in preds.items()}
    shared = y.notna() & preds[("Seq2Point", SEEDS[0])].notna() & preds[("NILMFormer", SEEDS[0])].notna()
    return y.where(shared), {k: v.where(shared) for k, v in preds.items()}


y_val, p_val = series_for("val", C.VAL_HOUSE)
SUMMARY["validation_shared_minutes"] = int(y_val.notna().sum())
rows = [{"model": m, "seed": s, **metrics.all_metrics(y_val, p)} for (m, s), p in p_val.items()]
val = pd.DataFrame(rows)
val.to_csv(C.TABLE_DIR / "nb07_validation_per_seed.csv", index=False)
cols = ["nde", "cycle_f1", "mae_w", "epd_wh", "f1", "sae", "overlap_pct", "extra_pct"]
agg = val.groupby("model")[cols].agg(["mean", "std"]).round(3)
print("shared validation minutes:", SUMMARY["validation_shared_minutes"])
agg

In [ ]:
m = val.groupby("model")[["nde", "cycle_f1"]].mean()
sd = val.groupby("model")[["nde", "cycle_f1"]].std()
margin = sd.max()  # the larger of the two models' seed SDs, per metric
better_nde = m.loc["NILMFormer", "nde"] < m.loc["Seq2Point", "nde"] - margin["nde"]
better_f1 = m.loc["NILMFormer", "cycle_f1"] > m.loc["Seq2Point", "cycle_f1"] + margin["cycle_f1"]
DECISION = "NILMFormer replaces Seq2Point" if (better_nde and better_f1) else "Seq2Point stays"
SUMMARY["validation"] = {
    "mean": m.round(4).to_dict(orient="index"), "sd": sd.round(4).to_dict(orient="index"),
    "margin": margin.round(4).to_dict(), "nilmformer_better_nde": bool(better_nde), "nilmformer_better_cycle_f1": bool(better_f1),
    "decision": DECISION,
}
print(f"NDE        Seq2Point {m.loc['Seq2Point','nde']:.3f}  NILMFormer {m.loc['NILMFormer','nde']:.3f}  margin {margin['nde']:.3f}  -> NILMFormer clearly better: {better_nde}")
print(f"cycle F1   Seq2Point {m.loc['Seq2Point','cycle_f1']:.3f}  NILMFormer {m.loc['NILMFormer','cycle_f1']:.3f}  margin {margin['cycle_f1']:.3f}  -> NILMFormer clearly better: {better_f1}")
print("DECISION:", DECISION)

In [ ]:
# secondary view: three-seed ensembles (mean power), the form the service uses
ens_val = {name: pd.concat([p_val[(name, s)] for s in SEEDS], axis=1).mean(axis=1, skipna=False) for name in ("Seq2Point", "NILMFormer")}
ens_val_tab = pd.DataFrame({name: metrics.all_metrics(y_val, p) for name, p in ens_val.items()}).T[cols].round(3)
SUMMARY["validation_ensemble"] = ens_val_tab.to_dict(orient="index")
ens_val_tab

## 4. Unseen test homes, scored once after the decision

Both three-seed ensembles on House 8 (the reference REFIT test home) and the five other unseen
homes, on the minutes both can score. These homes played no part in any choice above.

In [ ]:
test_rows, test_series = [], {}
for h in [C.TEST_HOUSE] + C.EXTRA_UNSEEN_HOUSES:
    y_h, p_h = series_for("test", h)
    for name in ("Seq2Point", "NILMFormer"):
        ens = pd.concat([p_h[(name, s)] for s in SEEDS], axis=1).mean(axis=1, skipna=False)
        test_rows.append({"house": h, "model": name, **metrics.all_metrics(y_h, ens)})
        if h == C.TEST_HOUSE:
            test_series[name] = ens
    if h == C.TEST_HOUSE:
        test_series["truth"] = y_h
test = pd.DataFrame(test_rows)
test.to_csv(C.TABLE_DIR / "nb07_unseen_ensembles.csv", index=False)
h8 = test[test.house == C.TEST_HOUSE].set_index("model")[cols].round(3)
mean6 = test.groupby("model")[cols].mean().round(3)
SUMMARY["house8"] = h8.to_dict(orient="index")
SUMMARY["unseen_mean"] = mean6.to_dict(orient="index")
SUMMARY["unseen_cycle_f1_per_home"] = test.pivot(index="house", columns="model", values="cycle_f1").round(3).to_dict()
print("House 8"); display(h8)
print("Mean over six unseen homes"); display(mean6)
test.pivot(index="house", columns="model", values=["nde", "cycle_f1"]).round(3)

In [ ]:
days = json.load(open(C.METRIC_DIR / "nb06_summary.json"))["plot_days"]
fig, axes = plt.subplots(len(days), 1, figsize=(12, 3.2 * len(days)))
for ax, day in zip(np.atleast_1d(axes), days):
    sl = slice(pd.Timestamp(day, tz="UTC"), pd.Timestamp(day, tz="UTC") + pd.Timedelta("1D"))
    ax.fill_between(test_series["truth"][sl].index, test_series["truth"][sl], color="0.8", label="plug monitor (truth)")
    ax.plot(test_series["Seq2Point"][sl], lw=1.2, label="Seq2Point ensemble")
    ax.plot(test_series["NILMFormer"][sl], lw=1.2, label="NILMFormer ensemble")
    ax.set(title=f"House 8, {day} (UTC)", ylabel="W")
    ax.legend(loc="upper right")
fig.tight_layout()
plots.save(fig, "nb07_house8_days")
plt.show()

## 5. Cost to serve

CPU time to process one day of 1-minute data (1,440 target minutes, one window each), single
thread, PyTorch eager mode, one model. This is the quantity the service pays for every request.

In [ ]:
torch.set_num_threads(1)
timing = {}
for name, path, loader in [("Seq2Point", f"m2_seq2point_w{W_S2P}_s42.pt", T.load_run), ("NILMFormer", f"m5_nilmformer_l{W_NF}_s42.pt", S.load_run)]:
    model = loader(C.MODEL_DIR / path, device="cpu")[0]
    win = W_S2P if name == "Seq2Point" else W_NF
    xb = torch.randn(1440, win) if name == "Seq2Point" else torch.randn(1440, 9, win)
    with torch.no_grad():
        model(xb[:64])
        t0 = time.perf_counter()
        for i in range(0, 1440, 256):
            model(xb[i : i + 256])
        timing[name] = time.perf_counter() - t0
SUMMARY["cpu_seconds_per_day_single_model"] = {k: round(v, 3) for k, v in timing.items()}
pd.Series(timing, name="CPU s per day of data (1 thread, one model)").round(2)

In [ ]:
(C.METRIC_DIR / "nb07_summary.json").write_text(json.dumps(SUMMARY, indent=2, default=float))
print(json.dumps(SUMMARY["validation"], indent=1))

## Summary

*(written after the run; see the outputs above)*
